# Iniciando o Spark

In [19]:
## Bloco de codigo para instalar versao especifica dos pacotes
!pip install pyspark

In [20]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [21]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Trusted_base_dados_cadastrais") \
    .getOrCreate()

# Importando bibliotecas

In [22]:
import os
import pytz
import datetime
from datetime import datetime
#from pyspark.sql.types import *
#from pyspark.sql.functions import count, avg
#import sys
#import numpy as np
#from datetime import datetime
#from pyspark.sql import SQLContext
#from datetime import timedelta
#from datetime import date
#from dateutil.relativedelta import relativedelta
#from pyspark.sql.functions import udf, lpad, translate

# Funções auxiliares e variáveis

In [1]:
# Função de log
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') + " >>>"

# Timestamp de processamento (com hora/minuto/segundo)
agora = datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc = agora.strftime("%Y%m%d%H%M%S")

# Data da execução (AAAAmmdd)
PROCESS_DATE = datetime.now().strftime("%Y%m%d")

# Período de referência (AAAAmm)
REF_PERIOD = datetime.now().strftime("%Y%m")

#Alterar o path_padrao caso seus arquivos não estejam nesse mesmo caminho
path_padrao = "/content/gdrive/Othercomputers/Meu laptop"

# Buckets e nomes de saída
bucket_base = "base_dados_cadastrais"
bucket_raw = f"{path_padrao}/Database_raw/base_dados_cadastrais"
bucket_trusted = f"{path_padrao}/Database_trusted/base_dados_cadastrais"
bucket_control = f"{path_padrao}/Database_control/base_dados_cadastrais"
output_trusted = f"trusted_{bucket_base}"

# Prints para conferência
print("PROCESS_DATE:", PROCESS_DATE)
print("REF_PERIOD:", REF_PERIOD)
print("dthproc:", dthproc)
print("bucket_raw:", bucket_raw)
print("bucket_trusted:", bucket_trusted)
print("bucket_control:", bucket_control)


NameError: name 'datetime' is not defined

# Importando os dados

In [24]:
path_raw = bucket_raw
df_raw = spark.read.parquet(path_raw)
df_raw.createOrReplaceTempView("df_Trusted_base_dados_cadastrais")

print(log(), "Registros na Raw:", df_raw.count())
df_raw.show(5, truncate=False)

2026-02-10 12:15:49 >>> Registros na Raw: 3900378
+-----------+------+---------------+----+----+---------+--------+----------------+------+------+------+------+------+------+------+------+------+------+----------+----------+------+------+------+------+------+--------+------+------------+------------+------+----------+-------------------------+-------------+
|NUM_CPF    |SAFRA |FLAG_INSTALACAO|FPD |PROD|flag_mig2|STATUSRF|DATADENASCIMENTO|var_03|var_02|var_04|var_05|var_06|var_07|var_08|var_09|var_10|var_11|var_12    |var_13    |var_14|var_15|var_16|var_17|var_18|var_19  |var_20|var_21      |var_22      |var_23|var_24    |var_25                   |CEP_3_digitos|
+-----------+------+---------------+----+----+---------+--------+----------------+------+------+------+------+------+------+------+------+------+------+----------+----------+------+------+------+------+------+--------+------+------------+------------+------+----------+-------------------------+-------------+
|77789989YZZ|202503|

# Processamento

In [25]:
#Alterando o datatype da base
df_base_dados_cadastrais = spark.sql(f"""
    SELECT

        CAST(NUM_CPF AS STRING) AS NUM_CPF,
        '{dthproc}' AS ts_proc,
        '{dthproc}' AS ts_proc_partition,
        CAST(SAFRA AS INT) AS SAFRA,
        CAST(SUBSTRING(CAST(SAFRA AS STRING), 1, 4) AS INT) AS SAFRA_ANO,
        CAST(SUBSTRING(CAST(SAFRA AS STRING), 5, 2) AS INT) AS SAFRA_MES,

        CAST(FLAG_INSTALACAO AS BOOLEAN) AS FLAG_INSTALACAO,
        CAST(FPD AS BOOLEAN) AS FPD,
        CAST(PROD AS STRING) AS PROD,
       CAST(flag_mig2 AS STRING) AS ProductMigration,
        CAST(STATUSRF AS STRING) AS STATUSRF,
        to_date(DATADENASCIMENTO, 'dd/MM/yyyy') AS DATA_DE_NASCIMENTO,
        to_date(var_12, 'dd/MM/yyyy') AS var_12,

        CAST(var_02 AS INT) AS var_02,
        CAST(var_03 AS INT) AS var_03,
        CAST(var_04 AS INT) AS var_04,
        CAST(var_05 AS INT) AS var_05,
        CAST(var_06 AS INT) AS var_06,
        CAST(var_07 AS FLOAT) AS var_07,
        CAST(var_08 AS INT) AS var_08,
        CAST(var_09 AS INT) AS var_09,
        CAST(var_10 AS STRING) AS Profissao,
        CAST(var_11 AS FLOAT) AS var_11,
        CAST(var_13 AS STRING) AS var_13,
        CAST(var_14 AS INT) AS var_14,
        CAST(var_15 AS STRING) AS Estado,
        CAST(var_16 AS INT) AS var_16,
        CAST(var_17 AS INT) AS var_17,
        CAST(var_18 AS STRING) AS var_18,
        CAST(var_19 AS STRING) AS var_19,
        CAST(var_20 AS STRING) AS var_20,
        CAST(var_21 AS STRING) AS var_21,
        CAST(var_22 AS STRING) AS Cargo,
        CAST(var_23 AS STRING) AS var_23,
        CAST(var_24 AS STRING) AS var_24,
        CAST(var_25 AS STRING) AS Tipo_de_Auxilio,
        CAST(CEP_3_digitos AS STRING) AS CEP_3_digitos

    FROM df_Trusted_base_dados_cadastrais

""")
df_base_dados_cadastrais.createOrReplaceTempView("lake_dados_cadastrais")
df_base_dados_cadastrais.cache()

print(log(), "Registros Trusted:", df_base_dados_cadastrais.count())
#df_base_dados_cadastrais.printSchema()
df_base_dados_cadastrais.show(5, truncate=False)

2026-02-10 12:15:52 >>> Registros Trusted: 3900378
+-----------+--------------+-----------------+------+---------+---------+---------------+-----+----+----------------+--------+------------------+----------+------+------+------+------+------+------+------+------+---------+------+----------+------+------+------+------+------+--------+------+------------+------------+------+----------+-------------------------+-------------+
|NUM_CPF    |ts_proc       |ts_proc_partition|SAFRA |SAFRA_ANO|SAFRA_MES|FLAG_INSTALACAO|FPD  |PROD|ProductMigration|STATUSRF|DATA_DE_NASCIMENTO|var_12    |var_02|var_03|var_04|var_05|var_06|var_07|var_08|var_09|Profissao|var_11|var_13    |var_14|Estado|var_16|var_17|var_18|var_19  |var_20|var_21      |Cargo       |var_23|var_24    |Tipo_de_Auxilio          |CEP_3_digitos|
+-----------+--------------+-----------------+------+---------+---------+---------------+-----+----+----------------+--------+------------------+----------+------+------+------+------+------+------

# Salvar na camada Trusted

In [26]:
# Salvar tabela no bucket na camada Trusted
path_trusted = os.path.join(bucket_trusted, output_trusted)
print("Trusted path:", path_trusted)

df_base_dados_cadastrais.write \
    .partitionBy("SAFRA","ts_proc_partition") \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(path_trusted)

Trusted path: /content/gdrive/Othercomputers/Meu laptop/Database_trusted/base_dados_cadastrais/trusted_base_dados_cadastrais


# Controle de carga

In [27]:
controle = spark.sql(f"""
    SELECT
        '{output_trusted}' AS name_file,
        ts_proc,
        ts_proc_partition,
        COUNT(*) AS qtd_registros
    FROM lake_dados_cadastrais
    GROUP BY 1,2,3
""")

controle.createOrReplaceTempView("controle")
controle.cache()

print(log(), "Registros controle:", controle.count())
controle.show(truncate=False)

2026-02-10 12:18:06 >>> Registros controle: 1
+-----------------------------+--------------+-----------------+-------------+
|name_file                    |ts_proc       |ts_proc_partition|qtd_registros|
+-----------------------------+--------------+-----------------+-------------+
|trusted_base_dados_cadastrais|20260210091549|20260210091549   |3900378      |
+-----------------------------+--------------+-----------------+-------------+



# Controle de processamento

In [28]:
path_control = os.path.join(bucket_control, f'tb_controle_processamento_{bucket_base}_trusted')
print("Control path:", path_control)

controle.write \
    .mode("append") \
    .option("compression", "snappy") \
    .parquet(path_control)

Control path: /content/gdrive/Othercomputers/Meu laptop/Database_control/base_dados_cadastrais/tb_controle_processamento_base_dados_cadastrais_trusted
